# Image Tracking Demo - Interactive Notebook

This notebook demonstrates spot tracking on 2D images using multiple tracking methods:
- **Gaussian Fit**: High-precision 2D Gaussian curve fitting
- **Parabola Fit**: 2D parabolic curve fitting
- **PyTrack**: Weighted centroid method

We'll create a test image, track a spot using all methods, and visualize the results with quantitative analysis.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from image_tracker import create_demo_image, track_spot

# Set up plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully!")

## 1. Create Test Image

We'll create a 100x100 pixel image with a single Gaussian spot at the center.

In [ ]:
# Create test image with centered spot
true_position = (50.0, 50.0)
image = create_demo_image(
    size=100,
    spot_center=true_position,
    spot_amplitude=100.0,
    spot_sigma=2.0
)

# Display image information
print(f"Image Shape: {image.shape}")
print(f"Value Range: [{image.min():.2f}, {image.max():.2f}]")
print(f"Mean: {image.mean():.2f}, Std: {image.std():.2f}")
print(f"True Spot Position: {true_position}")

In [ ]:
# Visualize the test image
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full image view
im1 = ax1.imshow(image, cmap='hot', origin='lower')
ax1.plot(true_position[0], true_position[1], 'g+', markersize=20, markeredgewidth=3, label='True Position')
ax1.set_title('Test Image (100x100 pixels)', fontsize=14, fontweight='bold')
ax1.set_xlabel('X (pixels)')
ax1.set_ylabel('Y (pixels)')
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.colorbar(im1, ax=ax1, label='Intensity')

# Zoomed view
zoom_range = 15
x_min, x_max = int(true_position[0] - zoom_range), int(true_position[0] + zoom_range)
y_min, y_max = int(true_position[1] - zoom_range), int(true_position[1] + zoom_range)
zoomed = image[y_min:y_max, x_min:x_max]

im2 = ax2.imshow(zoomed, cmap='hot', origin='lower', extent=[x_min, x_max, y_min, y_max])
ax2.plot(true_position[0], true_position[1], 'g+', markersize=15, markeredgewidth=2, label='True Position')
ax2.set_title('Zoomed View (±15 pixels)', fontsize=14, fontweight='bold')
ax2.set_xlabel('X (pixels)')
ax2.set_ylabel('Y (pixels)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.colorbar(im2, ax=ax2, label='Intensity')

plt.tight_layout()
plt.show()

## 2. Track Spot Using All Methods

Let's track the spot using all three available methods and compare their performance.

In [ ]:
# Track with all methods
methods = ["gaussian", "parabola", "pytrack"]
results = {}

print("Tracking spot with all methods:\n")
print("-" * 80)

for method in methods:
    result = track_spot(image, method=method, initial_guess=true_position)
    results[method] = result
    
    if result["success"]:
        error = np.sqrt((result['x'] - true_position[0])**2 + (result['y'] - true_position[1])**2)
        print(f"{method.upper():10s} -> Position: ({result['x']:.4f}, {result['y']:.4f}), Error: {error:.6f} px")
        if 'r_squared' in result:
            print(f"             R² = {result['r_squared']:.6f}")
    else:
        print(f"{method.upper():10s} -> FAILED: {result.get('error', 'Unknown error')}")
    print()

print("-" * 80)

## 3. Quantitative Analysis

Let's create a detailed comparison table and visualizations.

In [ ]:
# Create quantitative comparison table
import pandas as pd

successful_results = {k: v for k, v in results.items() if v.get("success", False)}

# Build comparison data
comparison_data = []
for method, result in successful_results.items():
    x_error = result['x'] - true_position[0]
    y_error = result['y'] - true_position[1]
    total_error = np.sqrt(x_error**2 + y_error**2)
    
    comparison_data.append({
        'Method': method.capitalize(),
        'X Position': f"{result['x']:.4f}",
        'Y Position': f"{result['y']:.4f}",
        'X Error': f"{x_error:.4f}",
        'Y Error': f"{y_error:.4f}",
        'Total Error (px)': f"{total_error:.6f}",
        'R²': f"{result.get('r_squared', 'N/A'):.6f}" if 'r_squared' in result else 'N/A'
    })

df = pd.DataFrame(comparison_data)
print("\n" + "=" * 100)
print("QUANTITATIVE COMPARISON TABLE")
print("=" * 100)
print(df.to_string(index=False))
print("=" * 100)

## 4. Visualization: Tracked Positions Overlay

In [ ]:
# Plot tracked positions on image
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {'gaussian': 'cyan', 'parabola': 'yellow', 'pytrack': 'magenta'}
markers = {'gaussian': 'x', 'parabola': '^', 'pytrack': 'o'}

# Full image
im1 = ax1.imshow(image, cmap='hot', origin='lower')
ax1.plot(true_position[0], true_position[1], 'g+', markersize=20, markeredgewidth=3, label='True Position')

for method, result in successful_results.items():
    ax1.plot(result['x'], result['y'], markers[method], color=colors[method],
             markersize=12, markeredgewidth=2, label=f'{method.capitalize()}')

ax1.set_title('Tracked Positions - Full View', fontsize=14, fontweight='bold')
ax1.set_xlabel('X (pixels)')
ax1.set_ylabel('Y (pixels)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
plt.colorbar(im1, ax=ax1, label='Intensity')

# Zoomed view
im2 = ax2.imshow(zoomed, cmap='hot', origin='lower', extent=[x_min, x_max, y_min, y_max])
ax2.plot(true_position[0], true_position[1], 'g+', markersize=15, markeredgewidth=2, label='True Position')

for method, result in successful_results.items():
    ax2.plot(result['x'], result['y'], markers[method], color=colors[method],
             markersize=10, markeredgewidth=2, label=f'{method.capitalize()}')

ax2.set_title('Tracked Positions - Zoomed View', fontsize=14, fontweight='bold')
ax2.set_xlabel('X (pixels)')
ax2.set_ylabel('Y (pixels)')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
plt.colorbar(im2, ax=ax2, label='Intensity')

plt.tight_layout()
plt.show()

## 5. Error Analysis

In [ ]:
# Create error analysis plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

methods_list = list(successful_results.keys())

# 1. Total error comparison
ax1 = axes[0, 0]
errors = []
for method in methods_list:
    result = successful_results[method]
    error = np.sqrt((result['x'] - true_position[0])**2 + (result['y'] - true_position[1])**2)
    errors.append(error)

bars = ax1.bar(range(len(methods_list)), errors,
               color=[colors.get(m, 'gray') for m in methods_list],
               edgecolor='black', linewidth=2)
ax1.set_xticks(range(len(methods_list)))
ax1.set_xticklabels([m.capitalize() for m in methods_list])
ax1.set_ylabel('Position Error (pixels)', fontweight='bold')
ax1.set_title('Total Position Error', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bar, error in zip(bars, errors):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{error:.6f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. X and Y error components
ax2 = axes[0, 1]
x_errors = [successful_results[m]['x'] - true_position[0] for m in methods_list]
y_errors = [successful_results[m]['y'] - true_position[1] for m in methods_list]

x_pos = np.arange(len(methods_list))
width = 0.35

ax2.bar(x_pos - width/2, x_errors, width, label='X Error', color='steelblue', edgecolor='black')
ax2.bar(x_pos + width/2, y_errors, width, label='Y Error', color='coral', edgecolor='black')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([m.capitalize() for m in methods_list])
ax2.set_ylabel('Error (pixels)', fontweight='bold')
ax2.set_title('X and Y Error Components', fontsize=12, fontweight='bold')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

# 3. R² comparison
ax3 = axes[1, 0]
r2_methods = [m for m in methods_list if 'r_squared' in successful_results[m]]
if r2_methods:
    r2_values = [successful_results[m]['r_squared'] for m in r2_methods]
    bars = ax3.bar(range(len(r2_methods)), r2_values,
                   color=[colors.get(m, 'gray') for m in r2_methods],
                   edgecolor='black', linewidth=2)
    ax3.set_xticks(range(len(r2_methods)))
    ax3.set_xticklabels([m.capitalize() for m in r2_methods])
    ax3.set_ylabel('R² Score', fontweight='bold')
    ax3.set_title('Goodness of Fit (R²)', fontsize=12, fontweight='bold')
    ax3.set_ylim([0, 1.05])
    ax3.grid(axis='y', alpha=0.3)
    
    for bar, r2 in zip(bars, r2_values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                 f'{r2:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Error scatter plot
ax4 = axes[1, 1]
for method in methods_list:
    result = successful_results[method]
    x_err = result['x'] - true_position[0]
    y_err = result['y'] - true_position[1]
    ax4.scatter(x_err, y_err, s=200, marker=markers.get(method, 'o'),
               color=colors.get(method, 'gray'), edgecolor='black', linewidth=2,
               label=method.capitalize())

ax4.axhline(y=0, color='gray', linestyle='--', linewidth=1)
ax4.axvline(x=0, color='gray', linestyle='--', linewidth=1)
ax4.plot(0, 0, 'g+', markersize=20, markeredgewidth=3, label='Perfect (0,0)')
ax4.set_xlabel('X Error (pixels)', fontweight='bold')
ax4.set_ylabel('Y Error (pixels)', fontweight='bold')
ax4.set_title('Error Scatter Plot', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.axis('equal')

plt.tight_layout()
plt.show()

## 6. Method Ranking and Summary

In [ ]:
# Rank methods by accuracy
print("\n" + "=" * 80)
print("METHOD RANKING BY ACCURACY (Best to Worst)")
print("=" * 80)

errors_dict = {}
for method, result in successful_results.items():
    error = np.sqrt((result['x'] - true_position[0])**2 + (result['y'] - true_position[1])**2)
    errors_dict[method] = error

sorted_methods = sorted(errors_dict.items(), key=lambda x: x[1])

for rank, (method, error) in enumerate(sorted_methods, 1):
    result = successful_results[method]
    r2_str = f", R² = {result['r_squared']:.6f}" if 'r_squared' in result else ""
    print(f"{rank}. {method.upper():<10s} - Error: {error:.6f} pixels{r2_str}")

print("=" * 80)

# Print recommendation
best_method = sorted_methods[0][0]
print(f"\n✓ RECOMMENDATION: Use '{best_method.upper()}' method for highest accuracy!")
print(f"  Error: {sorted_methods[0][1]:.6f} pixels")

## 7. Test with Different Spot Position

Let's verify the tracking methods work well for off-center spots too.

In [ ]:
# Test with off-center spot
off_center_pos = (30.0, 70.0)
print(f"Testing with off-center spot at {off_center_pos}...\n")

image2 = create_demo_image(size=100, spot_center=off_center_pos, spot_amplitude=100.0, spot_sigma=2.5)

# Track with best method (Gaussian)
result2 = track_spot(image2, method="gaussian")

if result2["success"]:
    error2 = np.sqrt((result2['x'] - off_center_pos[0])**2 + (result2['y'] - off_center_pos[1])**2)
    print(f"✓ Gaussian tracking successful!")
    print(f"  True Position: ({off_center_pos[0]:.4f}, {off_center_pos[1]:.4f})")
    print(f"  Found Position: ({result2['x']:.4f}, {result2['y']:.4f})")
    print(f"  Error: {error2:.6f} pixels")
    print(f"  R² = {result2['r_squared']:.6f}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(image2, cmap='hot', origin='lower')
    ax.plot(off_center_pos[0], off_center_pos[1], 'g+', markersize=20, markeredgewidth=3, label='True Position')
    ax.plot(result2['x'], result2['y'], 'cx', markersize=15, markeredgewidth=2, label='Tracked (Gaussian)')
    ax.set_title('Off-Center Spot Tracking Test', fontsize=14, fontweight='bold')
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.colorbar(im, ax=ax, label='Intensity')
    plt.show()
else:
    print(f"✗ Tracking failed: {result2.get('error', 'Unknown error')}")

## Summary

This notebook demonstrated:
1. Creating test images with Gaussian spots
2. Tracking spots using three different methods
3. Quantitative comparison of tracking accuracy
4. Visualization of results and error analysis
5. Method ranking and recommendations

**Key Findings:**
- The Gaussian fitting method typically provides the highest accuracy (sub-pixel precision)
- All methods successfully track spots with varying degrees of accuracy
- The R² score indicates the quality of fit for curve-fitting methods
- The methods work reliably for both centered and off-center spots